# 06 — Rolling windows

**Phenomenological Growth Models Lab**

Notebook 05 asked how good *one* forecast was. This notebook asks the question a
forecaster actually faces: if you had run this model every week as the data arrived,
how would it have done — and would you have picked the right model at the time?

The tool is a **rolling window**. Fit the model to a fixed-length slice of the series,
forecast the next few weeks, score the forecast against what was later observed, then
slide the window forward one week and do it all again. Every window is an independent
analysis, with only the data a forecaster would have had at that moment.

Assumes [05 — Forecasting](05_forecasting.ipynb). Needs GrowthFit 0.3 or later.

## Setup

Same environment as the other notebooks. The first run downloads and precompiles the
dependencies; after that, starting up takes seconds.

In [ ]:
using Pkg

# Works whether the notebook was launched from the repository root or from notebooks/.
root = isfile("Project.toml") ? "." : ".."
Pkg.activate(root)
Pkg.instantiate()

println("Active project: ", Base.active_project())

In [ ]:
using GrowthFit
using Plots, Statistics, Printf, DelimitedFiles, Random

default(size = (760, 420), lw = 2, legend = :topright, framestyle = :box,
        guidefontsize = 10, titlefontsize = 11,
        left_margin = 4Plots.mm, bottom_margin = 4Plots.mm)

# This notebook runs many bootstraps, so threads matter more here than anywhere else
# in the lab. A Jupyter kernel does NOT inherit `-t`; see the README for the one-off
# `IJulia.installkernel` step. Results are identical either way.
println("GrowthFit ", pkgversion(GrowthFit), " loaded. Threads available: ", Threads.nthreads())
pkgversion(GrowthFit) < v"0.3" && error("This notebook needs GrowthFit 0.3 or later for rolling_windows.")
Threads.nthreads() == 1 && println("  (single-threaded — the runs below will be several times slower; see README)");

In [ ]:
function load_series(name)
    path = isfile(joinpath("data", name)) ? joinpath("data", name) :
           joinpath("..", "data", name)
    isfile(path) || error("Could not find $name. Run this notebook from inside the " *
                          "cloned repository so the data/ folder is reachable.")
    raw = readdlm(path, '\t', Float64)
    return raw[:, 1], raw[:, 2]          # time, incidence
end

timevect_all, ydata_all = load_series("jalisco_measles.txt")
tpeak = timevect_all[argmax(ydata_all)]

@printf("%d weeks, peak of %.0f cases in week %.0f\n",
        length(ydata_all), maximum(ydata_all), tpeak)

## Expanding versus rolling

Notebook 05's rolling-origin loop used an **expanding** window: every forecast was fitted
to all the data from week 1 up to the forecast origin. A **rolling** window keeps a fixed
length instead and drops the oldest week each time it moves.

Why throw data away? Two reasons.

- **The early weeks may not belong to the same curve.** Look back at the fits in
  notebook 01: no single growth model captures the low, flat start of this outbreak
  *and* the rapid climb that follows. A fixed window lets the model describe the
  recent dynamics without being dragged by the oldest ones.
- **It answers a cleaner question.** With a fixed window, every forecast is made from
  the same amount of information, so skill can be compared fairly across the epidemic.

Window `i` covers observations `i` to `i + windowsize − 1`. We use 20 weeks, forecast 4
ahead, and — to keep this notebook to a few minutes — run every second window with a
modest bootstrap. The exercises at the end run the full version.

In [ ]:
windowsize       = 20
forecast_horizon = 4
M                = 50                     # bootstrap replicates per window
n_windows        = length(ydata_all) - windowsize + 1
wins             = 1:2:n_windows          # every second window

@printf("%d windows possible; running %d of them\n", n_windows, length(wins))
@printf("first window: weeks %.0f–%.0f    last window: weeks %.0f–%.0f\n",
        timevect_all[first(wins)], timevect_all[first(wins) + windowsize - 1],
        timevect_all[last(wins)],  timevect_all[last(wins)  + windowsize - 1])

## One model, window by window

`rolling_windows` does the whole loop: fit, bootstrap, score the fit, score the forecast.
It prints one line per window as it goes. Expect a minute or two.

In [ ]:
rw_grm = rolling_windows(:grm, timevect_all, ydata_all;
                         windowsize = windowsize, forecast_horizon = forecast_horizon,
                         dist = :nb1, M = M, windows = wins, seed = 2026)

Each line is one window: the weeks it covers, its AICc, how well the model describes the
weeks it was fitted to (`calWIS`), how well it forecast the weeks that followed (`fcWIS`),
and how many bootstrap replicates converged.

The result is a `RollingWindowResult`. Index it like a vector — `rw_grm[1]` is the first
window, with its own `fit`, `boot`, and metrics — or pull out tables with the three
accessors below.

### How the parameters move

`parameter_trajectory` gives the bootstrap median and 95% interval for every parameter,
one row per window. Plot them against the last week each window could see.

In [ ]:
pt = parameter_trajectory(rw_grm)

ci(x, lo, hi) = (x .- lo, hi .- x)        # ribbon = distance below and above the median

p1 = plot(pt.t_end, pt.r, ribbon = ci(pt.r, pt.r_lo, pt.r_hi), fillalpha = 0.25,
          marker = :circle, yscale = :log10, label = "", title = "growth rate r",
          xlabel = "last week in window")
p2 = plot(pt.t_end, pt.p, ribbon = ci(pt.p, pt.p_lo, pt.p_hi), fillalpha = 0.25,
          marker = :circle, ylims = (0, 1.05), label = "", title = "deceleration p",
          xlabel = "last week in window")
p3 = plot(pt.t_end, pt.K, ribbon = ci(pt.K, pt.K_lo, pt.K_hi), fillalpha = 0.25,
          marker = :circle, yscale = :log10, label = "", title = "final size K",
          xlabel = "last week in window")
for p in (p1, p2, p3)
    vline!(p, [tpeak], ls = :dash, lc = :grey40, label = "")
end
plot(p1, p2, p3, layout = (1, 3), size = (1000, 340))

The dashed line is the peak week.

Look at the windows that end **before** the peak. The intervals are wide, and some of
them run into the edges of the search space — the fitting bounds are `r ≤ 20`, `p ≤ 1`,
`a ≤ 10`, and `K` is capped at a large multiple of the window's total cases. A parameter
pressed against its bound is the optimizer telling you the data cannot locate it.

`K` is the clearest case. Before the turnover there is nothing in the data that says
where growth will stop, so its upper interval runs off towards the cap. Once the window
contains the peak *and* some of the decline, `K` settles and its interval narrows sharply.
This is notebook 04's identifiability problem, seen one window at a time.

### Fitting well is not forecasting well

`skill_trajectory` gives the scores, one row per window.

In [ ]:
st = skill_trajectory(rw_grm)

plot(st.t_end, st.WIS_cal, marker = :circle, yscale = :log10,
     label = "calibration WIS (the weeks it was fitted to)",
     xlabel = "last week in window", ylabel = "WIS", title = "GRM: fit versus forecast",
     legend = :topleft)
plot!(st.t_end, st.WIS_fc, marker = :diamond,
      label = "forecast WIS (the $(forecast_horizon) weeks after)")
vline!([tpeak], ls = :dash, lc = :grey40, label = "peak week")

The two lines measure different things, and they disagree most around the peak. A window
that ends just before the turnover is fitted very well — the curve passes through the
recent climb — and then forecasts badly, because nothing it has seen says the climb is
about to stop. In-sample fit says nothing about what happens next. That is notebook 05's
first rule of honest reporting, here made visible across the whole epidemic.

The last window or two score fewer than four forecast weeks, because the series runs out.
`st.steps_scored` records how many.

## Four models, the same windows

Now the question a forecaster cares about: which model should I have used? Run the same
windows for the Generalized Growth Model, the Generalized Logistic, and Richards, reusing
the GRM run from above. This takes a few minutes.

In [ ]:
models = (:ggm, :glm, :rich, :grm)
runs = Dict{Symbol,RollingWindowResult}(:grm => rw_grm)

for m in models
    haskey(runs, m) && continue
    print("running $(MODEL_NAMES[m]) ... ")
    runs[m] = rolling_windows(m, timevect_all, ydata_all;
                              windowsize = windowsize, forecast_horizon = forecast_horizon,
                              dist = :nb1, M = M, windows = wins, seed = 2026,
                              verbose = false)
    println("done")
end

sts = Dict(m => skill_trajectory(runs[m]) for m in models)

### What AICc says, window by window

AICc only compares models fitted to the **same** data. So the comparison runs *across* a
row — four models on one window — never down a column, because each window is different
data. (Under NB1 the raw AICc values are large and negative. That is expected: the
likelihood leaves out a term that depends only on the data, which shifts every model on
the same window by the same amount. Only the differences matter.)

In [ ]:
println(" week │  ΔAICc:  GGM     GLM    RICH     GRM │ lowest AICc │ best forecast")
println("─"^76)
for k in eachindex(sts[:ggm].window)
    a = [sts[m].aicc[k] for m in models]
    d = a .- minimum(a)
    w = [sts[m].WIS_fc[k] for m in models]
    best_fc = all(isnan, w) ? "–" : MODEL_NAMES[models[argmin(replace(w, NaN => Inf))]]
    @printf("%5.0f │ %13.1f %7.1f %7.1f %7.1f │ %-11s │ %s\n",
            sts[:ggm].t_end[k], d[1], d[2], d[3], d[4],
            MODEL_NAMES[models[argmin(a)]], best_fc)
end

Look at the windows that end before the peak. GGM — the one model with no saturation
term — should have the lowest AICc there, with GLM and GRM sitting at almost exactly the
same distance behind it in every one of those windows. Those numbers are not a
coincidence. They are the AICc **penalty** for one and two extra parameters:

In [ ]:
penalty(k, n) = 2k + 2k * (k + 1) / (n - k - 1)

# Free parameters under NB1, counting α: GGM (r, p, α) = 3; GLM adds K; GRM adds K and a.
@printf("GLM − GGM penalty: %.2f\n", penalty(4, windowsize) - penalty(3, windowsize))
@printf("GRM − GGM penalty: %.2f\n", penalty(5, windowsize) - penalty(3, windowsize))

If the AICc gap equals the penalty, the extra parameters bought **no** improvement in
likelihood at all. That is what happens before the turnover: with no sign of saturation in
the window, `K` drifts off towards infinity, and GLM and GRM become GGM with extra
parameters that do nothing.

So before the peak, AICc *cannot* tell you that the epidemic is about to turn — it will
favour the model that assumes it never will. Watch for the first window whose last week
is *after* the peak. That is where the ranking changes: one falling observation is enough
for the data to locate `K`.

The last column is the one that matters for forecasting, and it tells a different story.

### Forecast skill across the epidemic

In [ ]:
plt = plot(xlabel = "last week in window", ylabel = "forecast WIS", yscale = :log10,
           title = "$(forecast_horizon)-week-ahead forecast skill", legend = :topright)
for m in models
    plot!(plt, sts[m].t_end, sts[m].WIS_fc, marker = :circle, label = MODEL_NAMES[m])
end
vline!(plt, [tpeak], ls = :dash, lc = :grey40, label = "peak week")
plt

In [ ]:
println("model   mean forecast WIS   forecast 95% cover   fitted-window 95% cover")
println("─"^74)
for m in models
    s = sts[m]
    ok = .!isnan.(s.WIS_fc)
    @printf("%-6s %14.1f %18.0f%% %22.0f%%\n", MODEL_NAMES[m],
            mean(s.WIS_fc[ok]), mean(s.Coverage95PI_fc[ok]), mean(s.Coverage95PI_cal))
end

Three things to take from these two outputs.

**GGM fits its windows about as well as the others and forecasts far worse.** Compare its
two coverage columns. It describes the weeks it was fitted to perfectly respectably, then
misses most of the weeks that follow. It has no way to turn over, so once the peak is in
the forecast weeks it keeps projecting growth. Fit quality is not forecast skill, and
here the gap is dramatic.

**No single saturating model wins everywhere.** Read down the "best forecast" column of
the AICc table: the winner among GLM, Richards and GRM changes as the window moves. Check
whether the most flexible model, GRM, wins on average. In our run of this notebook it did
not: Richards and the Generalized Logistic finished almost level, both ahead of GRM, whose
extra parameter cost more in estimation noise than it bought in shape.

**Every model's forecast intervals are too narrow.** Their coverage on fitted weeks should
be close to the nominal 95%, but on the weeks they forecast it falls well short. The
bootstrap accounts for observation noise and for uncertainty in the parameters, but not
for the possibility that the model itself is wrong — and the previous point shows that
the right model keeps changing. That is exactly the situation in which combining models
helps, and it is why forecast hubs report ensembles rather than single models (exercise 5).

With `M = 50` and only every second window, the averages above are noisy. Treat the
pattern as the result, not the second decimal place.

### Skill by horizon

`forecast_skill_by_step` keeps one row per window *and* forecast week, so you can see how
fast skill decays with horizon.

In [ ]:
println("mean forecast WIS by weeks ahead")
println("model  ", join([lpad("h=$h", 8) for h in 1:forecast_horizon]))
for m in models
    fs = forecast_skill_by_step(runs[m])
    vals = [mean(fs.WIS[fs.horizon_step .== h]) for h in 1:forecast_horizon]
    println(rpad(MODEL_NAMES[m], 7), join([lpad(@sprintf("%.1f", v), 8) for v in vals]))
end

Further horizons are harder, as expected — but notice the steps are not averaged over the
same windows. The last windows run out of data and score only one or two steps, and those
windows sit on the low, easy tail of the epidemic. That flatters the short horizons
slightly. A fair horizon comparison uses only windows that scored every step:
`fs.window` tells you which those are.

The averages are also noisy: each one rests on about ten windows. If a model scores
better four weeks ahead than three, that is not skill improving with horizon — one
well-placed forecast is enough to move a mean of ten. As in notebook 05, do not read a
trend into a handful of numbers.

## Saving the results

The package deliberately writes no files. Every accessor returns a plain table — a
NamedTuple of equal-length vectors — so saving is a few lines of base Julia, or one
`CSV.write(path, DataFrame(table))` if you prefer. Output goes to `output/`, which git
ignores.

In [ ]:
outdir = joinpath(root, "output", "rolling_windows")
mkpath(outdir)

function write_table(path, t)
    open(path, "w") do io
        println(io, join(keys(t), ","))
        for i in eachindex(first(t))
            println(io, join((getfield(t, k)[i] for k in keys(t)), ","))
        end
    end
end

for m in models
    write_table(joinpath(outdir, "$(m)_skill.csv"),      skill_trajectory(runs[m]))
    write_table(joinpath(outdir, "$(m)_params.csv"),     parameter_trajectory(runs[m]))
    write_table(joinpath(outdir, "$(m)_fc_by_step.csv"), forecast_skill_by_step(runs[m]))
end
readdir(outdir)

## What to take from this

- **Evaluate a model the way you will use it.** One fit to the whole series tells you how
  well a curve describes an epidemic after the fact. Only a rolling evaluation tells you
  whether it would have helped at the time.
- **AICc cannot see a turning point that is not in the data yet.** Before the peak it
  prefers models that assume growth never stops. Choose models for forecasting on
  forecast skill.
- **In-sample coverage is not forecast coverage.** A model can bracket 95% of the weeks it
  was fitted to and a fraction of the weeks it forecasts.
- **When the best model keeps changing, combine them.**

---

## Exercises

1. **Window length.** Re-run with `windowsize = 15` and `windowsize = 25`. Shorter windows
   adapt faster and estimate parameters less precisely. Which effect wins for forecast
   WIS around the peak?

2. **The full run.** Remove `windows = wins` and set `M = 300`, for one model only. It
   takes considerably longer — use the timing of the run above to estimate how much before
   you start. Do the averages change, or only get smoother?

3. **Error structure.** Repeat the four-model comparison with `dist = :poisson`. Poisson
   assumes variance equals the mean. What happens to forecast coverage, and why?

4. **Another outbreak.** Load `cdmx_measles.txt` instead. It is shorter, so you will need
   a shorter window. Does the same model come out ahead?

5. **A simple ensemble.** For each window, pool the forecast samples of GLM, Richards and
   GRM — `runs[m][k].boot.forecast_noisy`, where row `windowsize + h` is `h` weeks
   ahead — into one predictive distribution and score it
   with the `wis` function from notebook 05. Does the equal-weight ensemble beat its best
   component on average? Does its coverage improve?

**Back:** [05 — Forecasting](05_forecasting.ipynb) · [Lab overview](../README.md)